In [1]:
import sys
import unittest.mock as mock

# Must happen BEFORE any other imports or pip installs
sys.modules['google.cloud.aiplatform'] = mock.MagicMock()

import os
import warnings
import subprocess
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore", message="RNN module weights are not part of single contiguous chunk")
warnings.filterwarnings("ignore", message="'pin_memory' argument is set as true but no accelerator")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
    "google-cloud-aiplatform",
    "google-cloud-storage",
], check=False)

IN_COLAB = os.path.exists("/content")
_packages = [
    "google-cloud-storage==2.10.0",
    "firebase-admin==6.3.0",
    "easyocr",
    "pymupdf",
    "bitsandbytes",
    "accelerate",
    "qdrant-client",
    "sentence-transformers",
    "kagglehub",
    "transformers==5.5.0",
]
_pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    # Colab has CUDA torch pre-installed — protect it from being downgraded
    _pip_cmd += ["--upgrade-strategy", "only-if-needed"]
subprocess.run(_pip_cmd + _packages, check=True)

print("✓ Dependencies installed")

Found existing installation: google-cloud-aiplatform 1.138.0
Uninstalling google-cloud-aiplatform-1.138.0:
  Successfully uninstalled google-cloud-aiplatform-1.138.0
Found existing installation: google-cloud-storage 3.9.0
Uninstalling google-cloud-storage-3.9.0:
  Successfully uninstalled google-cloud-storage-3.9.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.2/120.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 120.9 MB/s eta 0:00:00
✓ Dependencies installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-aiplatform[agent-engines]<2.0.0,>=1.132.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-adk 1.25.1 requires google-cloud-storage<4.0.0,>=2.18.0, but you have google-cloud-storage 2.10.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
 
import os
import gc
import json
import time
import uuid
import traceback
import re
import psutil
import datetime
from concurrent.futures import ThreadPoolExecutor
 
import torch
import fitz  # PyMuPDF
 
import firebase_admin
from firebase_admin import credentials, firestore, storage as fb_storage
from google.cloud.firestore_v1.base_query import FieldFilter
 
import easyocr
# import typesense
 
from transformers import pipeline, AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    PayloadSchemaType
)
import kagglehub
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

 
print("✓ Imports complete")

✓ Imports complete


In [3]:
# ── Gemma 4 model ────────────────────────────────────────────────────────────
# Gemma 4 26B MoE (called "27b-it" on Kaggle) — activates only ~4B params
# at inference time, fits comfortably on T4 at 4-bit.
# Kaggle model handle: google/gemma-4/transformers/gemma-4-27b-it
GEMMA_KAGGLE_HANDLE = "google/gemma-4/transformers/gemma-4-e4b-it"
GEMMA_BATCH_SIZE    = 4
# ── Embedding model ───────────────────────────────────────────────────────────
EMBED_MODEL_NAME    = "intfloat/multilingual-e5-large"
VECTOR_DIM          = 1024   # multilingual-e5-large output dimension
 
# ── Collection names ──────────────────────────────────────────────────────────
QDRANT_COLLECTION   = "scripture_pages"
TS_COLLECTION       = "scripture_pages"
 
# ── Memory thresholds ─────────────────────────────────────────────────────────
GPU_VRAM_MIN_FREE_GB = 0.5   # stop if GPU free VRAM drops below this
RAM_MIN_FREE_GB      = 3.0   # stop if system RAM free drops below this
 
# ── Safety caps ───────────────────────────────────────────────────────────────
MAX_BOOKS_PER_RUN    = 10    # hard ceiling per daily run
PAGES_PER_GCS_FLUSH  = 50    # flush partial GCS backup every N pages
 
# ── Stale lock timeout ────────────────────────────────────────────────────────
STALE_HOURS          = 6     # reset "processing" books older than this
 
print("✓ Configuration set")
 

✓ Configuration set


In [4]:
from google.oauth2 import service_account as _svc_acct
from google.cloud import firestore as _gfs

try:
    # Kaggle
    secrets = UserSecretsClient()
    def _get(name): return secrets.get_secret(name)
    print("✓ Secrets: Kaggle")
except Exception:
    try:
        # Google Colab
        from google.colab import userdata as _colab_ud
        def _get(name): return _colab_ud.get(name)
        print("✓ Secrets: Colab")
    except Exception:
        # Local: ~/ask-aagam-secrets.json
        _secrets_path = os.path.expanduser("~/ask-aagam-secrets.json")
        with open(_secrets_path) as _sf:
            _local_secrets = json.load(_sf)
        def _get(name): return _local_secrets[name]
        print("✓ Secrets: local file")

BUCKET_NAME = _get("GCS_BUCKET_NAME")
GCP_PROJECT = _get("GCP_PROJECT_ID")

sa_info = json.loads(_get("GCP_SERVICE_ACCOUNT_JSON"))

# Write SA to disk so ALL Google libs (storage, gRPC) pick it up via env var
_sa_path = "/tmp/gcp_sa.json"
with open(_sa_path, "w") as _f:
    json.dump(sa_info, _f)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = _sa_path

# ── Firebase Admin (for Storage bucket) ──────────────────────────────────────
cred = credentials.Certificate(sa_info)
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {
        "storageBucket": BUCKET_NAME,
        "projectId":     GCP_PROJECT,
    })
bucket = fb_storage.bucket()
print("✓ Firebase Admin + Storage initialised")

# ── Firestore: use google-cloud-firestore directly with explicit SA creds ─────
# firebase_admin.firestore.client() routes token refresh through Kaggle's gRPC
# credential plugin which tries metadata.google.internal → 300s timeout.
# Passing google.oauth2.service_account.Credentials explicitly bypasses that.
_sa_creds = _svc_acct.Credentials.from_service_account_info(
    sa_info,
    scopes=[
        "https://www.googleapis.com/auth/cloud-platform",
        "https://www.googleapis.com/auth/datastore",
    ],
)
db = _gfs.Client(project=GCP_PROJECT, credentials=_sa_creds)

# Fast-fail: verify auth works NOW before spending GPU time
try:
    list(db.collection("scriptures").limit(1).get())
    print("✓ Firestore connection verified")
except Exception as _e:
    raise RuntimeError(f"Firestore auth failed: {_e}") from _e

# ── Qdrant ────────────────────────────────────────────────────────────────────
qdrant = QdrantClient(
    url=_get("QDRANT_URL"),
    api_key=_get("QDRANT_API_KEY"),
    timeout=30,
)

print("✓ All service clients connected")
 
# ── Typesense ─────────────────────────────────────────────────────────────────
# ts_client = typesense.Client({
#     "nodes": [{
#         "host":     secrets.get_secret("TYPESENSE_HOST"),
#         "port":     "443",
#         "protocol": "https",
#     }],
#     "api_key":                    secrets.get_secret("TYPESENSE_API_KEY"),
#     "connection_timeout_seconds": 15,
# })
 
# ── Qdrant ────────────────────────────────────────────────────────────────────
qdrant = QdrantClient(
    url=_get("QDRANT_URL"),
    api_key=_get("QDRANT_API_KEY"),
    timeout=30,
)
 
print("✓ All service clients connected")

✓ All service clients connected


In [5]:
# ── Typesense ─────────────────────────────────────────────────────────────────
# try:
#     ts_client.collections[TS_COLLECTION].retrieve()
#     print(f"✓ Typesense collection '{TS_COLLECTION}' already exists")
# except typesense.exceptions.ObjectNotFound:
#     ts_client.collections.create({
#         "name": TS_COLLECTION,
#         "fields": [
#             {"name": "book_id",     "type": "string",   "facet": True},
#             {"name": "book_title",  "type": "string",   "facet": True},
#             {"name": "page_number", "type": "int32"},
#             {"name": "text",        "type": "string"},
#             {"name": "categories",  "type": "string[]", "facet": True},
#         ],
#     })
#     print(f"✓ Typesense collection '{TS_COLLECTION}' created")
 
# # ── Qdrant ────────────────────────────────────────────────────────────────────
# existing_collections = [c.name for c in qdrant.get_collections().collections]
# if QDRANT_COLLECTION not in existing_collections:
#     qdrant.create_collection(
#         collection_name=QDRANT_COLLECTION,
#         vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
#     )
#     qdrant.create_payload_index(QDRANT_COLLECTION, "book_id",    PayloadSchemaType.KEYWORD)
#     qdrant.create_payload_index(QDRANT_COLLECTION, "categories", PayloadSchemaType.KEYWORD)
#     print(f"✓ Qdrant collection '{QDRANT_COLLECTION}' created")
# else:
#     print(f"✓ Qdrant collection '{QDRANT_COLLECTION}' already exists")
 
 

In [6]:

 
HAS_GPU = torch.cuda.is_available()
HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE   = "cuda" if HAS_GPU else ("mps" if HAS_MPS else "cpu")
if not HAS_GPU:
    GEMMA_BATCH_SIZE = 1
print(f"\n  GPU: {HAS_GPU}  MPS: {HAS_MPS}  device: {DEVICE}  batch_size: {GEMMA_BATCH_SIZE}")

print(f"\nDownloading Gemma 4 from Kaggle: {GEMMA_KAGGLE_HANDLE}")
gemma_model_path = kagglehub.model_download(GEMMA_KAGGLE_HANDLE)
print(f"  Model path: {gemma_model_path}")

print("Loading Gemma 4 processor...")
gemma_processor = AutoProcessor.from_pretrained(gemma_model_path)

if HAS_GPU:
    gc.collect()
    torch.cuda.empty_cache()
    free_before = torch.cuda.mem_get_info(0)[0] / 1e9
    print(f"  GPU free before load: {free_before:.1f} GB")
    print("Loading Gemma 4 model (4-bit quantized, GPU)...")
    gemma_model = AutoModelForCausalLM.from_pretrained(
        gemma_model_path,
        device_map="auto",
        max_memory={0: "13GiB", "cpu": "48GiB"},
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        ),
    )
    used  = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU VRAM after load: {used:.1f}GB used / {total:.1f}GB total")
else:
    # 4-bit quant requires CUDA — bfloat16 on MPS (Apple Silicon) or CPU
    print(f"Loading Gemma 4 model (bfloat16, {DEVICE})...")
    gemma_model = AutoModelForCausalLM.from_pretrained(
        gemma_model_path,
        device_map=DEVICE,
        torch_dtype=torch.bfloat16,
    )
    ram_used = psutil.Process().memory_info().rss / 1e9
    print(f"  RAM after load: {ram_used:.1f}GB used")

gemma_model.eval()
print("✓ Gemma 4 loaded")

# ── Embedding model (always CPU) ──────────────────────────────────────────────
print(f"\nLoading {EMBED_MODEL_NAME} (CPU)...")
embedder = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
print("✓ Embedder loaded")

# ── EasyOCR ───────────────────────────────────────────────────────────────────
print("\nLoading EasyOCR (Hindi + English)...")
reader = easyocr.Reader(["hi", "en"], gpu=HAS_GPU)
# Compact LSTM weights — unwrap DataParallel if present, cover detector too
for attr in ["recognizer", "detector"]:
    m = getattr(reader, attr, None)
    if m is None:
        continue
    root = m.module if hasattr(m, "module") else m
    for sub in root.modules():
        if hasattr(sub, "flatten_parameters"):
            sub.flatten_parameters()
print("✓ EasyOCR loaded")
 


  Model path: /kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1
Loading Gemma 4 processor...
Loading Gemma 4 model (4-bit quantized)...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

✓ Gemma 4 loaded

Loading intfloat/multilingual-e5-large...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Using CPU. Note: This module is much faster with a GPU.


✓ Embedder loaded

  GPU VRAM after model load: 11.6GB used / 15.6GB total

Loading EasyOCR (Hindi + English)...


Progress: |██████████████████████████████████████████████████| 100.0% Complete✓ EasyOCR loaded


In [7]:
 
try:
    corrections = [
        c.to_dict()
        for c in db.collection("corrections")
                   .order_by("timestamp", direction=_gfs.Query.DESCENDING)
                   .limit(20)
                   .stream()
    ]
    FEEDBACK_CONTEXT = "\n".join(
        f"OCR Error: {e['original_text']}\nCorrected: {e['corrected_text']}"
        for e in corrections
    ) if corrections else "(no corrections yet)"
    print(f"✓ Loaded {len(corrections)} correction example(s)")
except Exception as e:
    print(f"  ! Could not load corrections (timeout/network): {e}")
    FEEDBACK_CONTEXT = "(no corrections yet)"
    corrections = []
 
print(f"✓ Loaded {len(corrections)} user correction example(s) as feedback context")
 
 

  ! Could not load corrections (timeout/network): Timeout of 300.0s exceeded, last exception: 503 Getting metadata from plugin failed with error: Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Compute Engine Metadata server unavailable. Last exception: HTTPConnectionPool(host='metadata.google.internal', port=80): Max retries exceeded with url: /computeMetadata/v1/instance/service-accounts/default/?recursive=true (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7f134b79f980>: Failed to resolve 'metadata.google.internal' ([Errno -2] Name or service not known)"))
✓ Loaded 0 user correction example(s) as feedback context


In [8]:
 
def gpu_free_gb() -> float:
    if not torch.cuda.is_available():
        return 99.0
    free, _ = torch.cuda.mem_get_info(0)
    return free / 1e9
 
def ram_free_gb() -> float:
    return psutil.virtual_memory().available / 1e9
 
def memory_ok() -> bool:
    gpu_free = gpu_free_gb()
    ram_free = ram_free_gb()
    print(f"  [memory] GPU free: {gpu_free:.1f}GB  |  RAM free: {ram_free:.1f}GB")
    return gpu_free >= GPU_VRAM_MIN_FREE_GB and ram_free >= RAM_MIN_FREE_GB
 
def release_page_memory():
    """Light GC between pages."""
    gc.collect()
    torch.cuda.empty_cache()
 
def release_book_memory():
    """Heavier GC between books — includes a brief pause."""
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2)

In [9]:
def gemma(prompt: str, system: str = None, max_new_tokens: int = 1024) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    text = gemma_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs    = gemma_processor(text=text, return_tensors="pt").to(gemma_model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = gemma_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    raw_response = gemma_processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,   # skip special tokens directly here
    )

    return raw_response.strip()     # parse_response() removed — decode handles it
 

In [10]:
 
def set_status(book_id: str, status: str, extra: dict = None):
    data = {"status": status, **(extra or {})}
    db.collection("scriptures").document(book_id).update(data)
 
def write_pages_firestore(book_id: str, pages: list):
    """
    Batch-write pages into scriptures/{bookId}/pages subcollection.
    Firestore max batch size is 500; we stay at 400 to be safe.
    """
    pages_ref = (
        db.collection("scriptures")
          .document(book_id)
          .collection("pages")
    )
    batch = db.batch()
    count = 0
    for page in pages:
        batch.set(
            pages_ref.document(str(page["page_number"])),
            {"pageNumber": page["page_number"], "lines": page["lines"]},
        )
        count += 1
        if count >= 400:
            batch.commit()
            batch = db.batch()
            count = 0
    if count:
        batch.commit()
 

In [11]:
# def index_typesense(book_id: str, book_title: str,
#                     categories: list, pages: list):
#     documents = [
#         {
#             "id":          f"{book_id}_p{p['page_number']}",
#             "book_id":     book_id,
#             "book_title":  book_title,
#             "page_number": p["page_number"],
#             "text":        "\n".join(p["lines"]),
#             "categories":  categories,
#         }
#         for p in pages
#     ]
#     CHUNK = 100
#     for i in range(0, len(documents), CHUNK):
#         ts_client.collections[TS_COLLECTION].documents.import_(
#             documents[i:i + CHUNK],
#             {"action": "upsert"},
#         )
 
 

In [12]:

def embed_and_upsert_qdrant(book_id: str, book_title: str,
                             categories: list, pages: list):
    """
    One vector per page. e5 models require a 'passage: ' prefix on documents.
    Batched at 32 to share VRAM safely with Gemma 4 and EasyOCR.
    """
    texts = [
        f"passage: {' '.join(p['lines'])}"
        for p in pages
    ]
 
    BATCH      = 32
    embeddings = []
    for i in range(0, len(texts), BATCH):
        vecs = embedder.encode(
            texts[i:i + BATCH],
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=BATCH,
        )
        embeddings.extend(vecs.tolist())
        release_page_memory()
 
    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=embeddings[idx],
            payload={
                "book_id":     book_id,
                "book_title":  book_title,
                "categories":  categories,
                "page_number": pages[idx]["page_number"],
                # 400-char preview so Next.js skips a second Firestore read
                "preview":     " ".join(pages[idx]["lines"])[:400],
            },
        )
        for idx in range(len(pages))
    ]
 
    UPSERT_CHUNK = 100
    for i in range(0, len(points), UPSERT_CHUNK):
        qdrant.upsert(
            collection_name=QDRANT_COLLECTION,
            points=points[i:i + UPSERT_CHUNK],
        )
 

In [ ]:
def write_all_outputs(book_id: str, book_title: str,
                      categories: list, all_pages: list) -> str:
    """Parallel write: Firestore pages + GCS JSON backup + Qdrant vectors."""
    final_gcs_path = f"processed/{book_id}.json"

    def _write_firestore():
        write_pages_firestore(book_id, all_pages)
        print(f"  ✓ Firestore pages written ({len(all_pages)} pages)")

    def _write_gcs():
        bucket.blob(final_gcs_path).upload_from_string(
            json.dumps(all_pages, ensure_ascii=False),
            content_type="application/json",
        )
        print(f"  ✓ GCS backup: gs://{BUCKET_NAME}/{final_gcs_path}")

    def _write_qdrant():
        embed_and_upsert_qdrant(book_id, book_title, categories, all_pages)
        print(f"  ✓ Qdrant vectors upserted")

    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = [
            ex.submit(_write_firestore),
            ex.submit(_write_gcs),
            ex.submit(_write_qdrant),
        ]
        for f in futures:
            f.result()  # re-raises on failure

    return final_gcs_path


In [13]:
def ocr_page(args):
    """Runs on CPU thread — extracts raw OCR for one page."""
    i, page, book_id = args
    pix      = page.get_pixmap(matrix=fitz.Matrix(2, 2))
    img_path = f"/tmp/{book_id}_p{i}.png"
    pix.save(img_path)
    raw_ocr  = " ".join(reader.readtext(img_path, detail=0))
    os.remove(img_path)
    return i, raw_ocr

def gemma_batch(prompts: list[str], max_new_tokens: int = 1024) -> list[str]:
    """
    Runs multiple prompts through Gemma in one batched forward pass.
    Requires left-padding for batch inference.
    """
    gemma_processor.tokenizer.padding_side = "left"

    texts = [
        gemma_processor.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for p in prompts
    ]

    inputs = gemma_processor(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(gemma_model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = gemma_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    results = []
    for i in range(len(prompts)):
        decoded = gemma_processor.decode(
            outputs[i][input_len:],
            skip_special_tokens=True,
        )
        results.append(decoded.strip())

    return results


In [14]:
OCR_SYSTEM_PROMPT = """You are an expert OCR refinement engine specialising in Jain Scriptures (Aagams).
Your only job is to take raw OCR text and return clean, high-fidelity Unicode Devanagari/English.
Never add commentary, never explain your changes, never greet the user.
Output ONLY the corrected text."""

META_SYSTEM_PROMPT = """You are an expert in Jain Scriptures and metadata extraction.
Return ONLY a valid JSON object — no markdown fences, no preamble, no explanation."""

from concurrent.futures import ThreadPoolExecutor

def process_book(book: dict):
    book_id   = book["id"]
    gcs_path  = book.get("gcsPath", "")
    file_name = gcs_path.replace("uploads/", "")

    print(f"\n  Marking '{book_id}' → processing")
    set_status(book_id, "processing", {
        "processingStartedAt": _gfs.SERVER_TIMESTAMP,
    })

    # ── Download PDF from GCS ─────────────────────────────────────────────────
    pdf_path = f"/tmp/{file_name}"

    # ── Resume: load partial progress from previous crashed run ──────────────
    partial_blob = bucket.blob(f"processed/partial/{book_id}.json")
    if partial_blob.exists():
        resumed_pages = json.loads(partial_blob.download_as_string())
        done_page_nums = {p["page_number"] for p in resumed_pages}
        print(f"  ↺ Resuming — {len(done_page_nums)} pages already done")
    else:
        resumed_pages = []
        done_page_nums = set()

    bucket.blob(f"uploads/{file_name}").download_to_filename(pdf_path)
    print(f"  ✓ Downloaded: {file_name}")

    # ── OCR only pages not yet refined ───────────────────────────────────────
    doc         = fitz.open(pdf_path)
    total_pages = len(doc)
    remaining   = [i for i in range(total_pages) if (i + 1) not in done_page_nums]
    print(f"  Running OCR on {len(remaining)}/{total_pages} pages (parallel GPU)...")

    ocr_args = [(i, doc.load_page(i), book_id) for i in remaining]
    with ThreadPoolExecutor(max_workers=2 if HAS_GPU else 1) as executor:
        ocr_results = list(executor.map(ocr_page, ocr_args))

    ocr_results.sort(key=lambda x: x[0])
    doc.close()
    os.remove(pdf_path)
    print(f"  ✓ OCR complete")

    # ── Gemma refinement (GPU only) / raw OCR save (CPU-only run) ───────────
    all_pages = list(resumed_pages)

    print(f"  Refining with Gemma (batch size={GEMMA_BATCH_SIZE})...")
    for batch_start in range(0, len(ocr_results), GEMMA_BATCH_SIZE):
        batch     = ocr_results[batch_start:batch_start + GEMMA_BATCH_SIZE]
        page_nums = [b[0] for b in batch]
        raw_texts = [b[1] for b in batch]

        prompts = [
            f"""LEARNED CORRECTIONS FROM USER FEEDBACK:
            {FEEDBACK_CONTEXT}
            
            REFINEMENT RULES:
            1. Decode legacy fonts (Krutidev, Shivaji, etc.) → Unicode Devanagari.
            2. Preserve matras, conjunct consonants (संयुक्ताक्षर), and anusvara exactly.
            3. Fix Jain terminology: Tirthankara, Anekantavada, Syadvada, Agama, etc.
            4. Keep English, numbers, and already-correct Unicode unchanged.
            5. Completely unresolvable text → [illegible]. Never hallucinate.
            
            RAW OCR INPUT:
            {raw}"""
            for raw in raw_texts
        ]

        refined_batch = gemma_batch(prompts, max_new_tokens=512)

        for idx, refined in enumerate(refined_batch):
            # Strip preamble/markdown Gemma sometimes echoes back
            cleaned = re.sub(r'\*\*(.*?)\*\*', r'\1', refined)
            cleaned = re.sub(r'^\s*CLEAN REFINED TEXT:\s*\n?', '', cleaned, flags=re.IGNORECASE).strip()
            all_pages.append({
                "page_number": page_nums[idx] + 1,
                "lines":       [l for l in cleaned.split("\n") if l.strip()],
            })

        if len(all_pages) > 0 and len(all_pages) % PAGES_PER_GCS_FLUSH == 0:
            bucket.blob(f"processed/partial/{book_id}.json").upload_from_string(
                json.dumps(all_pages, ensure_ascii=False),
                content_type="application/json",
            )
            print(f"    [flush] {len(all_pages)}/{total_pages} pages")

        release_page_memory()
        print(f"    pages {batch_start+1}–{min(batch_start+GEMMA_BATCH_SIZE, total_pages)}/{total_pages}")

    all_pages.sort(key=lambda x: x["page_number"])
    print(f"  ✓ Gemma refinement complete — {len(all_pages)} pages")

    # ── Metadata extraction — first 10 pages for better author detection ──────
    # First 5 pages (title/cover) + last 5 pages (colophon has author in Jain texts)
    front = all_pages[:5]
    back  = all_pages[-5:] if len(all_pages) > 5 else []
    meta_context = " ".join(" ".join(p["lines"]) for p in front + back)

    # Derive a filename hint (strip timestamp prefix and extension)
    fname_hint = re.sub(r"^\d+-", "", file_name).rsplit(".", 1)[0].replace("-", " ").replace("_", " ")

    meta_raw = gemma(
        system=META_SYSTEM_PROMPT,
        prompt=f"""You are an expert in Jain Scriptures (Aagams) and their classification.
        Extract metadata from the provided text.
        Return ONLY a valid JSON object — no markdown fences, no preamble.

        FILENAME HINT (use as strong signal for title if text is ambiguous): {fname_hint}
        
        Required keys:
          title        → string (full title of the scripture in its original language;
                         the FIRST page usually has the title prominently — use that, not a verse)
          writer       → string (original author/composer — two patterns:
                         BEFORE name: रचयिता, कर्ता, प्रणेता, लेखक, आचार्य, मुनि, कविवर, पं., श्री
                         AFTER name: कृत, विरचित, रचित, प्रणीत (e.g. "ध्यानतराय जी कृत")
                         Honorifics to strip: स्वर्गीय, पूज्य, श्री, जी
                         Also check the LAST pages (colophon). If truly not found, use "Unknown")
          tikakar      → array of strings (commentators — look for टीकाकार, टीका, अनुवादक,
                         संपादक before names. Empty array if none)
          categories   → array of strings — classify into one or more of these four Anuyogs ONLY:
                         "Karananuyog" (mathematics, cosmology, karma theory),
                         "Charananuyog" (conduct, ethics, vows, pratikraman),
                         "Dravyanuyog" (philosophy, metaphysics, soul, substance),
                         "Prathamanuyog" (narratives, biographies of Tirthankaras)
          description  → string (3–4 sentence summary: what the scripture covers,
                         its significance, and which tradition it belongs to)
        
        TEXT (first 5 pages + last 5 pages):
        {meta_context[:6000]}
        
        JSON:""",
        max_new_tokens=600,
    )

    try:
        s        = meta_raw.find("{")
        e        = meta_raw.rfind("}") + 1
        metadata = json.loads(meta_raw[s:e])
    except Exception:
        print(f"  ! Metadata JSON parse failed. Raw: {meta_raw[:300]}")
        metadata = {
            "title":       file_name,
            "writer":      "Unknown",
            "tikakar":     [],
            "categories":  [],
            "description": "",
        }

    book_title = metadata.get("title") or file_name
    categories = metadata.get("categories") or []
    print(f"  ✓ Metadata — title='{book_title}', writer='{metadata.get('writer')}', categories={categories}")

    # ── Write all outputs (Firestore + GCS parallel, then Qdrant) ────────────
    final_gcs_path = write_all_outputs(book_id, book_title, categories, all_pages)

    # ── Firestore root doc — metadata + ready status ──────────────────────────
    # Clean up partial file now that book is complete
    partial_blob.delete() if partial_blob.exists() else None

    set_status(book_id, "ready", {
        "title":               book_title,
        "pageCount":           len(all_pages),
        "gcsBackup":           f"gs://{BUCKET_NAME}/{final_gcs_path}",
        "processedAt":         _gfs.SERVER_TIMESTAMP,
        "processingStartedAt": _gfs.DELETE_FIELD,
        **{k: v for k, v in metadata.items() if k != "title"},
    })
    print(f"  ✓ Firestore root doc → ready")

In [15]:
import datetime
print("\n" + "═" * 70)
print("  STARTING PROCESSING RUN")
print("═" * 70)
 
# ── Reset stale "processing" locks ───────────────────────────────────────────
stale_docs = (
    db.collection("scriptures")
      .where(filter=FieldFilter("status", "==", "processing"))
      .stream()
)
for doc in stale_docs:
    data    = doc.to_dict()
    started = data.get("processingStartedAt")
    if started and hasattr(started, "tzinfo"):
        age_hours = (
            datetime.datetime.now(datetime.timezone.utc) - started
        ).total_seconds() / 3600
        if age_hours > STALE_HOURS:
            db.collection("scriptures").document(doc.id).update({
                "status":       "pending",
                "errorMessage": f"Reset: stuck in processing for {age_hours:.1f}h",
            })
            print(f"  ↺ Reset stale processing lock: {doc.id} ({age_hours:.1f}h old)")

# ── Retry "failed" books (re-queue on every run) ──────────────────────────────
failed_docs = (
    db.collection("scriptures")
      .where(filter=FieldFilter("status", "==", "failed"))
      .stream()
)
for doc in failed_docs:
    db.collection("scriptures").document(doc.id).update({
        "status":       "pending",
        "errorMessage": _gfs.DELETE_FIELD,
        "failedAt":     _gfs.DELETE_FIELD,
    })
    print(f"  ↺ Re-queued failed book: {doc.id}")
 
# ── Fetch pending queue ───────────────────────────────────────────────────────
pending = (
    db.collection("scriptures")
      .where(filter=FieldFilter("status", "==", "pending"))
      .limit(MAX_BOOKS_PER_RUN)
      .stream()
)
queue = [{"id": d.id, **d.to_dict()} for d in pending] 
if not queue:
    print("\n  No pending books in queue. Exiting cleanly.")
    raise SystemExit(0)
 
print(f"\n  {len(queue)} pending book(s) in queue")
print(f"  Processing sequentially until memory threshold or queue is empty.\n")
 
# ── Main loop ─────────────────────────────────────────────────────────────────
processed_count = 0
failed_count    = 0
 
for book in queue:
    print(f"\n{'─' * 70}")
    print(f"  Book [{processed_count + 1}/{len(queue)}]: {book['id']}")
    print(f"{'─' * 70}")
 
    # Memory gate — check before starting each book
    if not memory_ok():
        print(f"\n  ⚠ Memory threshold reached after {processed_count} book(s).")
        print(f"  {len(queue) - processed_count - failed_count} book(s) remain queued for tomorrow.")
        break
 
    try:
        process_book(book)
        processed_count += 1
    except Exception as exc:
        tb = traceback.format_exc()
        print(f"\n  ✗ Failed: {exc}")
        print(tb)
        try:
            db.collection("scriptures").document(book["id"]).update({
                "status":       "failed",
                "errorMessage": str(exc)[:500],
                "failedAt":     _gfs.SERVER_TIMESTAMP,
            })
        except Exception:
            pass
        failed_count += 1
    finally:
        # Always release memory between books, even on failure
        release_book_memory()
 


══════════════════════════════════════════════════════════════════════
  STARTING PROCESSING RUN
══════════════════════════════════════════════════════════════════════


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


RetryError: Timeout of 300.0s exceeded, last exception: 503 Getting metadata from plugin failed with error: Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Compute Engine Metadata server unavailable. Last exception: HTTPConnectionPool(host='metadata.google.internal', port=80): Max retries exceeded with url: /computeMetadata/v1/instance/service-accounts/default/?recursive=true (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7f134b79f9b0>: Failed to resolve 'metadata.google.internal' ([Errno -2] Name or service not known)"))

In [ ]:
remaining = len(queue) - processed_count - failed_count
 
print("\n" + "═" * 70)
print("  RUN COMPLETE")
print("═" * 70)
print(f"  Processed:  {processed_count} book(s)  ✓")
print(f"  Failed:     {failed_count} book(s)  ✗")
print(f"  Remaining:  {remaining} book(s) queued for next run")
print(f"  GPU free:   {gpu_free_gb():.1f} GB")
print(f"  RAM free:   {ram_free_gb():.1f} GB")
print("═" * 70)

In [ ]:
# ── Post-run verification ─────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  VERIFICATION")
print("═" * 70)

errors = []
for book_id in [b["id"] for b in queue[:processed_count]]:
    doc = db.collection("scriptures").document(book_id).get().to_dict()
    expected_pages = doc.get("pageCount", 0)

    # 1. Firestore pages subcollection
    fs_count = len(list(
        db.collection("scriptures").document(book_id).collection("pages").stream()
    ))

    # 2. GCS backup
    gcs_blob = bucket.blob(f"processed/{book_id}.json")
    gcs_ok = gcs_blob.exists()

    # 3. Qdrant vectors
    q_count = qdrant.count(
        collection_name=QDRANT_COLLECTION,
        count_filter={"must": [{"key": "book_id", "match": {"value": book_id}}]},
    ).count

    ok = (fs_count == expected_pages and gcs_ok and q_count == expected_pages)
    icon = "✓" if ok else "✗"
    print(f"  {icon} {book_id}")
    print(f"      Firestore pages : {fs_count}/{expected_pages}")
    print(f"      GCS backup      : {'exists' if gcs_ok else 'MISSING'}")
    print(f"      Qdrant vectors  : {q_count}/{expected_pages}")
    if not ok:
        errors.append(book_id)

if errors:
    print(f"\n  ✗ {len(errors)} book(s) incomplete: {errors}")
else:
    print(f"\n  ✓ All {processed_count} book(s) fully verified")
print("═" * 70)
